# Humor Genome Open Controls

### A deterministic causal-design lab for surprise, resolution, and over-explanation

> **Executive summary.** This notebook verifies and analyzes 120,000 openly reusable procedural controls. It does not claim that any row is funny, human-authored, culturally representative, or evidence about a brain mechanism.

| Question | Answer |
|---|---|
| What problem is addressed? | Existing joke corpora rarely contain matched alternatives that separate expected continuation, unresolved surprise, compact repair, and explicit explanation. |
| What is the proposed solution? | Generate four arms from the same premise and configuration, preserve their group IDs, isolate templates across splits, and publish every build/audit receipt. |
| What can this notebook conclude? | The release bytes, grouping, balance, and retrieval contract are reproducible. It can measure generator artifacts and benchmark models against declared relations. |
| What can it not conclude? | Human funniness, audience benefit, safety, originality worldwide, or neural surprise reduction. Those require different evidence. |

The corpus operationalizes the project hypothesis `expectation -> violation -> optional repair`. That is a falsifiable starting point, not a result. Source and methods: [aidonerightcorp/humorvibes-jestry](https://github.com/aidonerightcorp/humorvibes-jestry).


## 1. Verify the mounted release before reading it

Kaggle uploads can succeed while carrying stale or partial files. This cell discovers the dataset by its declared ID, verifies every SHA-256 and byte length, then loads the controlling summary.


In [ ]:
from pathlib import Path
import glob, hashlib, json, os

def find_release():
    candidates = [Path(p).parent for p in glob.glob('/kaggle/input/**/release_summary.json', recursive=True)]
    candidates += [Path('../kaggle_open_controls'), Path('kaggle_open_controls')]
    for root in candidates:
        try:
            summary = json.loads((root / 'release_summary.json').read_text(encoding='utf-8'))
        except (FileNotFoundError, json.JSONDecodeError):
            continue
        if summary.get('dataset_id') == 'humor-genome-open-controls':
            return root.resolve(), summary
    raise FileNotFoundError('Attach taylorsamarel/humor-genome-open-controls')

DATA_DIR, SUMMARY = find_release()
manifest = json.loads((DATA_DIR / 'manifest.json').read_text(encoding='utf-8'))
for name, expected in manifest['files'].items():
    path = DATA_DIR / name
    assert path.is_file(), f'missing manifest payload: {name}'
    digest = hashlib.sha256()
    with path.open('rb') as fh:
        while chunk := fh.read(1024 * 1024):
            digest.update(chunk)
    assert path.stat().st_size == expected['bytes'], f'byte mismatch: {name}'
    assert digest.hexdigest() == expected['sha256'], f'hash mismatch: {name}'
print('dataset:', DATA_DIR)
print('verified payload files:', len(manifest['files']))
print('generator commit:', SUMMARY['generator_commit'])
assert SUMMARY['human_authored_rows'] == 0
assert SUMMARY['human_rated_rows'] == 0


## 2. Load the controlled rows

The Parquet and JSONL files carry the same records. Parquet is used here for speed. The assertions below make the experimental unit and leakage boundary visible before any chart is drawn.


In [ ]:
import pandas as pd
rows = pd.read_parquet(DATA_DIR / 'open_controls.parquet')
assert len(rows) == SUMMARY['rows'] == 120_000
assert rows['item_id'].is_unique
assert rows.groupby('premise_id')['split'].nunique().max() == 1
assert rows.groupby('template_family_id')['split'].nunique().max() == 1
assert not rows['human_authored'].any()
assert not rows['human_rated'].any()
assert rows['funniness_label'].isna().all()

counts = rows.groupby(['counterfactual_arm', 'surface_variant']).size().unstack()
print(f"{len(rows):,} rows; {rows['premise_id'].nunique()} premise families; "
      f"{rows['template_family_id'].nunique()} isolated lexical templates")
display(counts)
display(rows.groupby('split').size().rename('rows').to_frame())


## 3. Inspect one matched group

All four arms below share a premise, slot configuration, split, and surface variant. Only the type of continuation changes. `intended_mechanism` describes how the generator was constructed; it is not an observed psychological label.


In [ ]:
example_id = sorted(rows['configuration_id'].unique())[0]
example = rows[(rows['configuration_id'] == example_id) & (rows['surface_variant'] == 0)]
display(example[['counterfactual_arm', 'setup', 'punchline', 'repair_type']].sort_values('counterfactual_arm'))


## 4. Adversarial artifact audit

A corpus can accidentally encode its labels through length or punctuation. The release builder groups rows by coarse surface signatures and asks how accurately the majority arm in each group predicts the label. Chance is 25%. This is intentionally a hostile diagnostic. Passing the release threshold does not mean the text is artifact-free.


In [ ]:
import re
from collections import Counter, defaultdict

surface = defaultdict(Counter)
for text, arm in zip(rows['text'], rows['counterfactual_arm']):
    words = re.findall(r'\b\w+\b', text)
    signature = (min(len(words)//5, 20), min(len(text)//30, 20), text.count('.'), text.count(','), text.count(':'))
    surface[signature][arm] += 1
accuracy = sum(max(group.values()) for group in surface.values()) / len(rows)
published_audit = json.loads((DATA_DIR / 'audit.json').read_text(encoding='utf-8'))
assert abs(accuracy - published_audit['adversarial']['surface_only_arm_accuracy']) < 1e-12
print(f'surface-only arm accuracy: {accuracy:.1%} (chance 25.0%)')
print('release threshold: <80%; status:', 'PASS' if accuracy < .80 else 'FAIL')
print('Important: residual predictability is a limitation to report, not a model-quality result.')


## 5. A fully executable retrieval baseline

The dataset includes one query and one relevant compact-repair document per premise. This baseline uses TF-IDF, evaluates each split independently, and reports MRR and Recall@k. The qrels are generator relations—not human relevance judgments—so this tests pipeline correctness and model sensitivity only.


In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

docs = pd.read_json(DATA_DIR / 'retrieval_documents.jsonl', lines=True)
queries = pd.read_json(DATA_DIR / 'retrieval_queries.jsonl', lines=True)
qrels = pd.read_json(DATA_DIR / 'retrieval_qrels.jsonl', lines=True)
truth = dict(zip(qrels.query_id, qrels.document_id))

def evaluate_split(split):
    d = docs[docs.split == split].reset_index(drop=True)
    q = queries[queries.split == split].reset_index(drop=True)
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)
    matrix = vectorizer.fit_transform(pd.concat([d.text, q.text], ignore_index=True))
    scores = cosine_similarity(matrix[len(d):], matrix[:len(d)])
    ranks = []
    for i, query in q.iterrows():
        order = np.argsort(-scores[i], kind='stable')
        relevant = truth[query.query_id]
        rank = int(np.where(d.document_id.to_numpy()[order] == relevant)[0][0]) + 1
        ranks.append(rank)
    return {'split': split, 'queries': len(ranks), 'MRR': sum(1/r for r in ranks)/len(ranks),
            'Recall@1': sum(r <= 1 for r in ranks)/len(ranks),
            'Recall@10': sum(r <= 10 for r in ranks)/len(ranks)}

retrieval_results = pd.DataFrame(evaluate_split(split) for split in ('train', 'validation', 'test'))
display(retrieval_results.round(4))
assert len(docs) == len(queries) == len(qrels) == 300


### Use another embedding model without changing the benchmark

Export document and query vectors in the existing row order, compute cosine similarity, and reuse the same qrels and split loop. The repository API already supports deterministic hash vectors, multiple allowlisted Ollama embedding models, OpenAI-compatible embedding endpoints, and optional sentence-transformers. Always report the exact provider, model revision, dimensions, normalization, and split. Embedding similarity is not proof of originality, equivalence, or funniness.


## 6. What useful conclusions require next

The next research release should preregister a blinded, randomized rating study and collect the supplied fields separately: expectedness, surprise, resolution, funniness, familiarity, comprehensibility, and offensiveness. Analyze people—not generated rows—as the independent unit, preserve writer/rater clustering, and hold premise families out. A useful finding would be an interaction such as compact resolution improving funniness relative to unresolved surprise, with uncertainty and audience context reported.

Until then, this corpus is already useful for deterministic application fixtures, grouped-split regression tests, retrieval bakeoffs, experimental-design teaching, and preparing a consented human study. It is not a shortcut around that study.


In [ ]:
receipt = {
    'dataset_id': SUMMARY['dataset_id'],
    'generator_commit': SUMMARY['generator_commit'],
    'manifest_files_verified': len(manifest['files']),
    'rows': len(rows),
    'premise_families': int(rows['premise_id'].nunique()),
    'surface_only_arm_accuracy': accuracy,
    'retrieval_baseline': retrieval_results.to_dict(orient='records'),
    'human_authored_rows': 0,
    'human_rated_rows': 0,
    'claim_ready_for_human_funniness': False,
    'status': 'VERIFIED_SYNTHETIC_CONTROL_RELEASE',
}
output_root = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path('.')
(output_root / 'OPEN_CONTROLS_NOTEBOOK_RECEIPT.json').write_text(
    json.dumps(receipt, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print(json.dumps(receipt, indent=2, sort_keys=True))
